In [4]:
# imports
import pandas as pd
import boto3
from io import StringIO

In [5]:
# get data from aws
client = boto3.client("s3")
bucket = "reddit-data-306915831561-us-east-2-an"
files = client.list_objects_v2(Bucket=bucket)["Contents"]
reddit_data = pd.concat([
    pd.read_csv(StringIO(client.get_object(Bucket=bucket, Key=f["Key"])["Body"].read().decode("utf-8")))
    for f in files
])

In [6]:
# combine title and body into 1 column
def combine_text(row):
    if pd.isna(row["selftext"]) or row["selftext"] in ("", "[deleted]", "[removed]", "Title."):
        return row["title"]
    return row["title"] + " | " + row["selftext"]
reddit_data["text"] = reddit_data.apply(lambda x: combine_text(x), axis=1)

In [7]:
# drop columns
reddit_data.drop(columns=["created_utc", "num_comments", "score", "created_at", "selftext", "title"], inplace=True)

In [8]:
# filter out deleted posts
reddit_data = reddit_data[reddit_data.text != "[deleted by user]"]

In [9]:
# randomly sample data for use
sample_pct = 0.1
sampled_data = reddit_data.groupby("year", group_keys=False).apply(lambda x: x.sample(frac=sample_pct, random_state=699))
sampled_data.head()

/var/folders/mj/jdg6nf_91m30_bh08_9683mr0000gn/T/ipykernel_55210/2037045512.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_data = reddit_data.groupby("year", group_keys=False).apply(lambda x: x.sample(frac=sample_pct, random_state=699))


,id,subreddit,year,month,text
1506,c1xgk4,cscareerquestions,2019,6,Best Leetcode List to go through without Premi...
392,aydng7,cscareerquestions,2019,3,NaN
2270,d3eb6t,cscareerquestions,2019,9,Summary of my job hunt as a recent CS grad | I...
78,afusg6,jobs,2019,1,NaN
1585,csch68,jobs,2019,8,How important is having a degree | I’m at univ...


In [10]:
data = reddit_data[~reddit_data.id.isin(sampled_data.id)]
annotate_data = data.groupby("year", group_keys=False).apply(lambda x: x.sample(min(len(x), 50), random_state=699))
annotate_data.head()

/var/folders/mj/jdg6nf_91m30_bh08_9683mr0000gn/T/ipykernel_55210/3501387277.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annotate_data = data.groupby("year", group_keys=False).apply(lambda x: x.sample(min(len(x), 50), random_state=699))


,id,subreddit,year,month,text
2055,ctsc7f,cscareerquestions,2019,8,What does agile/scrum experience even mean? | ...
1896,dg29m3,jobs,2019,10,Don't let overly ambitious recruiters give you...
1741,cehhxc,cscareerquestions,2019,7,"New CS grad. No internships, no experience, no..."
677,bccuc3,jobs,2019,4,How to keep going after so many rejection lett...
1586,c6ewkd,cscareerquestions,2019,6,"Summer interns of reddit, what have you alread..."


In [11]:
for year, group in annotate_data.groupby("year"):
    globals()[f"annotate_data_{year}"] = group
for year, group in annotate_data.groupby("year"):
    group.to_csv(f"annotate_data_{year}.csv", index=False)

In [15]:
annotated_2019 = pd.read_csv("annotated_data_2019.csv")
annotated_2020 = pd.read_csv("annotated_data_2020.csv")
annotated_2021 = pd.read_csv("annotated_data_2021.csv")
annotated_2022 = pd.read_csv("annotated_data_2022.csv")
annotated_2023 = pd.read_csv("annotated_data_2023.csv")
annotated_2024 = pd.read_csv("annotated_data_2024.csv")
annotated_2025 = pd.read_csv("annotated_data_2025.csv")
annotated_data = pd.concat([annotated_2019, annotated_2020, annotated_2021, annotated_2022, annotated_2023, annotated_2024, annotated_2025])
annotated_data.head()

,id,subreddit,year,month,text,sentiment
0,bs7mvu,jobs,2019,5,Why do hiring managers do this?,neg
1,d6imv9,jobs,2019,9,A simple way to make the job hunt a little mor...,neg
2,calny9,cscareerquestions,2019,7,So I just landed a work from home Software Dev...,neutral
3,eei3jk,cscareerquestions,2019,12,Applied for grad SE role. Got bumped up | As t...,pos
4,b8xk8s,cscareerquestions,2019,4,How to avoid making your coworkers life misera...,neutral


In [16]:
# put data in aws
client.put_object(Bucket=bucket, Key="sampled_data.csv", Body=sampled_data.to_csv(index=False))
client.put_object(Bucket=bucket, Key="annotated_data.csv", Body=annotated_data.to_csv(index=False))

{'ResponseMetadata': {'RequestId': '714368VK6HZNB5N5',
  'HostId': 'HaAsfZpFmKNbIq60QSl8HWlZ0ErgQnGTrOQIPHJJV7J2NWkmgQc0Yjka6SyaxsvVmfLNS+dU1+fvp488oS1FSLNi/C8hBhMr',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'HaAsfZpFmKNbIq60QSl8HWlZ0ErgQnGTrOQIPHJJV7J2NWkmgQc0Yjka6SyaxsvVmfLNS+dU1+fvp488oS1FSLNi/C8hBhMr',
   'x-amz-request-id': '714368VK6HZNB5N5',
   'date': 'Mon, 06 Apr 2026 03:29:43 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"8782873bf2b15a6a604c000ad80fd50b"',
   'x-amz-checksum-crc32': 'VPPw2w==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"8782873bf2b15a6a604c000ad80fd50b"',
 'ChecksumCRC32': 'VPPw2w==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}